# Figure 4

The 24 typography settings, one font per row and one vertical position band per column, each shown with one word image and its control image.

Inputs: `lexicon/block*.txt`; `scripts/stimuli/backgrounds.py`, `scripts/stimuli/render.py`.

In [ ]:
import itertools

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.font_manager import FontProperties
from matplotlib.patches import FancyBboxPatch
from matplotlib.textpath import text_to_path
from PIL import Image

from src import lexicon
from src.config import BACKGROUNDS, local_path
from src.slots import slot_table
from src.tau import manifest
from src.typography import FONT_FAMILIES, PLACEMENTS, font_px
from src.utils import setup_notebook, viz

setup_notebook()

IMAGE_PATHS = manifest().set_index(["word", "slot", "condition"]).image_path
SLOT_TABLE = slot_table().set_index("slot")
BACKGROUND_TABLE = pd.read_parquet(
    BACKGROUNDS / "backgrounds.parquet").set_index("background_index")

#: One row per typography setting, indexed by ``(font, placement)``.
APPEARANCE = (SLOT_TABLE.reset_index().drop_duplicates("appearance")
              .sort_values("appearance"))
CROSSING = APPEARANCE.set_index(["font", "placement"])

N_ROWS, N_COLUMNS = len(FONT_FAMILIES), len(PLACEMENTS)

#: Left to right, the four bands from the top of the image down.
COLUMN_ORDER = ("top", "upper_middle", "lower_middle", "bottom")
PLACEMENT_NAMES = {"top": "Top", "upper_middle": "Upper middle",
                   "lower_middle": "Lower middle", "bottom": "Bottom"}

print(f"typography settings: {len(APPEARANCE)} = {N_ROWS} fonts x "
      f"{N_COLUMNS} vertical position bands")

## Words and their cells

Within a row the longest word takes the smallest type, and of the 720 ways to give the six groups to the six fonts, the one with the lowest correlation between word length and type size is drawn.

In [ ]:
#: Six groups of ``(word, role)``, one group to a row. For the last two rows the
#: role is the word's category or industry.
GROUPS = (
    (("torch", "anchor"), ("flashlight", "synonym"), ("touch", "one-edit"),
     ("blork", "pseudoword")),
    (("brave", "anchor"), ("bold", "synonym"), ("cowardly", "antonym"),
     ("drith", "pseudoword")),
    (("unsafe", "negation"), ("risky", "synonym"), ("safe", "base"),
     ("droon", "pseudoword")),
    (("road", "singular"), ("roads", "plural"), ("italy", "country"),
     ("rome", "capital")),
    (("owl", "animal"), ("laptop", "device"), ("auditorium", "structure"),
     ("cactus", "plant")),
    (("toyota", "carmaker"), ("pepsi", "beverage"), ("nintendo", "video game"),
     ("costco", "retail chain")),
)

#: Cap height in pixels of every setting, by ``(font, placement)``.
CAP_PX = {key: font_px(row["size"]) for key, row in CROSSING.iterrows()}


def seat(group, font):
    """A row's words against that font's four settings, longest word smallest.

    Ties in length keep the written order, and ties in size keep the column order.
    """
    columns = sorted(COLUMN_ORDER, key=lambda p: (CAP_PX[font, p],
                                                  COLUMN_ORDER.index(p)))
    order = sorted(range(len(group)), key=lambda i: (-len(group[i][0]), i))
    return {column: group[i] for column, i in zip(columns, order)}


def running_against(seating):
    """Correlation of word length with cap height over the grid. Lower is better."""
    lengths, caps = [], []
    for font, row in seating.items():
        for placement, (word, _) in row.items():
            lengths.append(len(word))
            caps.append(CAP_PX[font, placement])
    return float(np.corrcoef(lengths, caps)[0, 1])


SEATING, BEST_SCORE = None, np.inf
for fonts in itertools.permutations(FONT_FAMILIES):
    seating = {font: seat(group, font) for font, group in zip(fonts, GROUPS)}
    if (score := running_against(seating)) < BEST_SCORE:
        SEATING, BEST_SCORE = {font: seating[font] for font in FONT_FAMILIES}, score

Every word drawn comes from one of the eight word lists, and its label is the role recorded there. A noun's or brand's role is its category or industry.

In [ ]:
BLOCKS = [pd.DataFrame(lexicon.read_block(f"block{i}.txt")) for i in range(1, 9)]
RECORDED = {}  # word: (list, role)
for number, block in enumerate(BLOCKS, start=1):
    for column in block.columns.drop(["class", "category", "ortho"], errors="ignore"):
        for row, word in block[column].items():
            RECORDED[word] = (number, block.category[row] if column == "word" else column)
SHOWN_AS = {"one-edit": "spelling_twin"}
drawn = [(word, SHOWN_AS.get(role, role)) for group in GROUPS for word, role in group]
lists = {RECORDED[word][0] for word, _ in drawn}
matched = sum(RECORDED[word][1] == role for word, role in drawn)
print(f"words drawn: {len(drawn)}, from {len(lists)} of the {len(BLOCKS)} lists; "
      f"label equal to the recorded role: {matched} of {len(drawn)}")

## Background in each cell

Each setting is rendered on four backgrounds, and a cell shows the one on which its word changes the scene most over the pixels its letters mark.

In [ ]:
#: A pixel counts as marked when the word changed it by more than this, of 255.
#: Below it lie the antialiased rim of a glyph and the edge of the panel.
INK_FLOOR = 12.0


def image(word: str, slot: int, condition: str) -> np.ndarray:
    path = local_path(IMAGE_PATHS[word, slot, condition])
    return np.asarray(Image.open(path).convert("RGB"), dtype=float)


def background(slot: int) -> np.ndarray:
    index = int(SLOT_TABLE.loc[slot, "background_index"])
    return np.asarray(Image.open(
        local_path(BACKGROUND_TABLE.loc[index, "path"])).convert("RGB"), dtype=float)


def ink_contrast(word: str, slot: int) -> float:
    """Mean change the letters made to the scene, over the pixels they marked."""
    delta = np.abs(image(word, slot, "word") - background(slot)).mean(axis=2)
    marked = delta > INK_FLOOR
    return float(delta[marked].mean()) if marked.any() else 0.0


def hardest(word: str, font: str, placement: str) -> int:
    """The slot of this setting on which the word marks the scene most."""
    appearance = int(CROSSING.loc[(font, placement), "appearance"])
    candidates = SLOT_TABLE.index[SLOT_TABLE.appearance.eq(appearance)]
    scored = {int(slot): ink_contrast(word, int(slot)) for slot in candidates}
    return max(scored, key=scored.get)


CELLS = {}
for font, row in SEATING.items():
    for placement, (word, role) in row.items():
        CELLS[font, placement] = dict(word=word, role=role,
                                      slot=hardest(word, font, placement))

## Geometry

The canvas is one axes in points, and the figure takes the full text width.

In [ ]:
WIDTH_PT = viz.TEXT_WIDTH_IN * viz.PT_PER_INCH

PAIR_GAP = 2.5        # a word image to its own control image
BLOCK_GAP = 10.0      # one cell to the next
LABEL_DROP = 7.0      # tile bottom to the baseline of the word beneath it
ROW_GAP = 5.5         # that baseline to the next row's tiles
HEAD_GAP = 4.0        # top row's tiles to the baseline of the head above them
HEAD_RISE = 8.5       # that baseline to the baseline of the column names
TOP_ROOM = 2.0
BOTTOM_ROOM = 2.5     # under the last baseline, for a descender

WORD_SIZE = 6.8
ROLE_SIZE = 5.8
ROLE_SPACE = 3.0      # the word's ink to its role's ink, on one baseline
COLUMN_SIZE = 7.0
HEAD_SIZE = 5.8
TILE_RADIUS = 0.8


def advance(text: str, size: float) -> float:
    """The width in points a string advances, the same at every dpi."""
    prop = FontProperties(family="Noto Sans", size=size)
    return text_to_path.get_text_width_height_descent(text, prop, ismath=False)[0]


#: The tiles take the whole text width.
TILE = (WIDTH_PT - (N_COLUMNS - 1) * BLOCK_GAP
        - N_COLUMNS * PAIR_GAP) / (2 * N_COLUMNS)

BLOCK = 2 * TILE + PAIR_GAP
ROW_PITCH = TILE + LABEL_DROP + ROW_GAP
HEIGHT_PT = (BOTTOM_ROOM + N_ROWS * ROW_PITCH - ROW_GAP + LABEL_DROP + HEAD_GAP
             + HEAD_SIZE + HEAD_RISE + TOP_ROOM)

#: Left edge of each cell, and the bottom of each row's tiles.
X_CELL = {placement: column * (BLOCK + BLOCK_GAP)
          for column, placement in enumerate(COLUMN_ORDER)}
Y_ROW = {font: BOTTOM_ROOM + LABEL_DROP + (N_ROWS - 1 - row) * ROW_PITCH
         for row, font in enumerate(FONT_FAMILIES)}

## Figure

In [ ]:
def canvas(width_pt, height_pt):
    fig = plt.figure(figsize=(width_pt / viz.PT_PER_INCH, height_pt / viz.PT_PER_INCH))
    ax = fig.add_axes((0, 0, 1, 1))
    ax.set_xlim(0, width_pt)
    ax.set_ylim(0, height_pt)
    ax.set_aspect("equal")
    ax.set_axis_off()
    return fig, ax


def rounded(ax, x, y, w, h, face, *, r, edge="none", lw=0.6, z=1.0):
    patch = FancyBboxPatch((x, y), w, h,
                           boxstyle=f"round,pad=0,rounding_size={r}",
                           facecolor=face, edgecolor=edge, lw=lw, zorder=z)
    ax.add_patch(patch)
    return patch


def tile(ax, img, x, y, s, *, r=TILE_RADIUS, z=2.0):
    """An image in a square with slightly rounded corners, lower-left at (x, y)."""
    art = ax.imshow(img, extent=(x, x + s, y, y + s), zorder=z,
                    interpolation="lanczos")
    art.set_clip_path(rounded(ax, x, y, s, s, "none", r=r, z=z))
    rounded(ax, x, y, s, s, "none", r=r, edge=viz.RULE, lw=0.4, z=z + 0.1)


def label(ax, x, y, text, *, size, color=viz.INK, ha="center", va="baseline"):
    return ax.text(x, y, text, fontsize=size, color=color, ha=ha, va=va)


def cell_label(ax, centre, y, word, role):
    """The word in ink and its role in grey after it, centred together on one baseline."""
    width = advance(word, WORD_SIZE) + ROLE_SPACE + advance(role, ROLE_SIZE)
    start = centre - width / 2
    label(ax, start, y, word, size=WORD_SIZE, ha="left")
    label(ax, start + advance(word, WORD_SIZE) + ROLE_SPACE, y, role,
          size=ROLE_SIZE, color=viz.CONTROL, ha="left")


def draw_figure():
    fig, ax = canvas(WIDTH_PT, HEIGHT_PT)
    top = Y_ROW[FONT_FAMILIES[0]] + TILE
    for placement in COLUMN_ORDER:
        label(ax, X_CELL[placement] + BLOCK / 2, top + HEAD_GAP + HEAD_RISE,
              PLACEMENT_NAMES[placement], size=COLUMN_SIZE, color=viz.INK)
    first = X_CELL[COLUMN_ORDER[0]]
    label(ax, first + TILE / 2, top + HEAD_GAP, "Word image", size=HEAD_SIZE,
          color=viz.CONTROL)
    label(ax, first + TILE + PAIR_GAP + TILE / 2, top + HEAD_GAP, "Control image",
          size=HEAD_SIZE, color=viz.CONTROL)

    for font in FONT_FAMILIES:
        y = Y_ROW[font]
        for placement in COLUMN_ORDER:
            x, cell = X_CELL[placement], CELLS[font, placement]
            tile(ax, image(cell["word"], cell["slot"], "word").astype(np.uint8),
                 x, y, TILE)
            tile(ax, image(cell["word"], cell["slot"], "scramble").astype(np.uint8),
                 x + TILE + PAIR_GAP, y, TILE)
            cell_label(ax, x + BLOCK / 2, y - LABEL_DROP, cell["word"], cell["role"])
    return fig

In [ ]:
fig = draw_figure()
viz.save(fig, "figure4_rendering")